# Factory-to-Customer Shipping Route Efficiency Analysis for Nassau Candy Distributor

## Notebook 01: Project Overview and Data Loading

### Project Background
Nassau Candy Distributor operates as a national distributor, shipping products from factories to customers across multiple US regions in the United States. In such logistics operations, shipping efficiency directly affects customer satisfaction, operational cost, and business scalability.

Despite having detailed order and shipment data, the company currently lacks route-level visibility into shipping performance. Without this visibility, logistics decisions remain reactive instead of data-driven.

### Problem Statement
The organization currently lacks clarity on:
- Which factory-to-customer routes are consistently efficient
- Which routes experience frequent delays
- How shipping performance varies by region, state, and ship mode
- Where operational bottlenecks exist geographically

### Project Goal
The main goal of this project is to transform raw order and shipment data into route-level logistics intelligence. This project will help identify efficient routes, delay-prone routes, regional bottlenecks, and ship mode performance patterns.

### Final Deliverables
- Cleaned and engineered logistics dataset
- Jupyter Notebook analysis
- Streamlit dashboard
- Business insights and recommendations
- Research paper / case study documentation

## Analytical Methodology

This project will follow the official step-by-step methodology:

1. **Data Cleaning & Validation**
   - Validate date formats
   - Remove invalid or negative lead times
   - Handle missing shipment records
   - Standardize geographic fields

2. **Feature Engineering**
   - Calculate Shipping Lead Time
   - Create Factory → Customer Region routes
   - Create Factory → Customer State routes
   - Group shipments by ship mode

3. **Route Definition & Aggregation**
   - Calculate total shipments by route
   - Calculate average lead time by route
   - Measure route lead time variability

4. **Efficiency Benchmarking**
   - Rank fastest to slowest routes
   - Identify top 10 efficient routes
   - Identify bottom 10 inefficient routes
   - Compare route performance by ship mode

5. **Geographic Bottleneck Analysis**
   - Identify high lead-time regions
   - Detect high-volume poor-performance areas
   - Detect congestion-prone states and regions

6. **Ship Mode Performance Analysis**
   - Compare shipping efficiency across ship modes
   - Evaluate descriptive cost-time tradeoffs

## Dataset Description

The raw dataset contains order, shipment, customer, geographic, product, and financial information.

### Expected key columns:
- Order and shipment dates
- Ship mode
- Customer and location details
- Region and state information
- Product details
- Sales, units, cost, and gross profit

### Important note
At the raw-data stage, the dataset may contain quality issues such as:
- inconsistent date formats
- unrealistic shipment dates
- invalid lead times
- missing values
- inconsistent geographic labels

These issues will be handled in the next notebook during the data cleaning and validation stage.

In [1]:
# Import core libraries
import pandas as pd
import numpy as np
import statistics as stats
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Raw Dataset

In this step, we load the original Nassau Candy Distributor raw dataset from the project folder.

The analysis in this notebook is based only on the raw source file. No cleaned dataset is used here.

In [4]:
#Load raw dataset
file_path = pd.read_csv(r"C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Nassau Candy Distributor Logistics Project\data\Nassau Candy Distributor.csv")
ncd = pd.DataFrame(file_path)
print("Raw dataset loaded successfully.")
print(ncd)

Raw dataset loaded successfully.
       Row ID                      Order ID  Order Date   Ship Date       Ship Mode  Customer ID Country/Region           City        State/Province Postal Code   Division    Region     Product ID                       Product Name  Sales  Units  Gross Profit  Cost
0           1  US-2021-103800-CHO-MIL-31000  03-01-2024  30-06-2026  Standard Class       103800  United States        Houston                 Texas       77095  Chocolate  Interior  CHO-MIL-31000         Wonka Bar - Milk Chocolate   6.50      2          4.22  2.28
1           2  US-2021-112326-CHO-TRI-54000  04-01-2024  01-07-2026  Standard Class       112326  United States     Naperville              Illinois       60540  Chocolate  Interior  CHO-TRI-54000  Wonka Bar - Triple Dazzle Caramel   7.50      2          4.90  2.60
2           3  US-2021-112326-CHO-NUT-13000  04-01-2024  01-07-2026  Standard Class       112326  United States     Naperville              Illinois       60540  Chocola

In [5]:
# Preview first 5 rows
ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [6]:
# Check dataset shape
print(f"Number of rows: {ncd.shape[0]}")
print(f"Number of columns: {ncd.shape[1]}")

Number of rows: 10194
Number of columns: 18


In [7]:
# View all column names
ncd.columns.tolist()

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Country/Region',
 'City',
 'State/Province',
 'Postal Code',
 'Division',
 'Region',
 'Product ID',
 'Product Name',
 'Sales',
 'Units',
 'Gross Profit',
 'Cost']

## Initial Structural Inspection

Now we inspect:
- data types
- missing values
- column structure
- overall raw dataset readiness

This step helps us understand what cleaning tasks will be required in the next notebook.

In [8]:
# Dataset information
ncd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

In [9]:
# Check missing values
missing_values = ncd.isnull().sum().sort_values(ascending=False)
missing_values

Row ID          0
Order ID        0
Order Date      0
Ship Date       0
Ship Mode       0
               ..
Product Name    0
Sales           0
Units           0
Gross Profit    0
Cost            0
Length: 18, dtype: int64

In [10]:
# Check missing value percentage
missing_percent = (ncd.isnull().sum() / len(ncd) * 100).sort_values(ascending=False)
missing_percent

Row ID          0.0
Order ID        0.0
Order Date      0.0
Ship Date       0.0
Ship Mode       0.0
               ... 
Product Name    0.0
Sales           0.0
Units           0.0
Gross Profit    0.0
Cost            0.0
Length: 18, dtype: float64

In [11]:
# Basic descriptive statistics for numerical columns
ncd.describe()

,Row ID,Customer ID,Sales,Units,Gross Profit,Cost
count,10194.000000,10194.000000,10194.000000,10194.000000,10194.000000,10194.000000
mean,5097.500000,134468.961154,13.908537,3.791838,9.166451,4.742087
std,2942.898656,20231.483007,11.341020,2.228317,6.643740,5.061647
min,1.000000,100006.000000,1.250000,1.000000,0.250000,0.600000
25%,2549.250000,117212.000000,7.200000,2.000000,4.900000,2.400000
50%,5097.500000,133550.000000,10.800000,3.000000,7.470000,3.600000
75%,7645.750000,152051.000000,18.000000,5.000000,12.250000,5.700000
max,10194.000000,192314.000000,260.000000,14.000000,130.000000,130.000000


In [12]:
# Basic descriptive statistics for object columns
ncd.describe(include="object")

,Order ID,Order Date,Ship Date,Ship Mode,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name
count,10194,10194,10194,10194,10194,10194,10194,10194,10194,10194,10194,10194
unique,8549,717,1338,4,2,542,59,654,3,4,15,15
top,US-2022-112130-CHO-FUD-51000,02-09-2025,07-06-2028,Standard Class,United States,New York City,California,10035,Chocolate,Pacific,CHO-MIL-31000,Wonka Bar - Milk Chocolate
freq,5,62,38,6120,9994,915,2001,263,9844,3253,2137,2137


In [13]:
# Check duplicate rows
duplicate_count = ncd.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


In [14]:
# Preview random sample of rows for manual inspection
ncd.sample(5, random_state=42)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
6636,6637,US-2023-108777-CHO-FUD-51000,12-12-2025,03-06-2029,Same Day,108777,United States,Lorain,Ohio,44052,Chocolate,Atlantic,CHO-FUD-51000,Wonka Bar - Fudge Mallows,10.80,3,7.20,3.6
3742,3743,US-2022-102778-CHO-TRI-54000,21-11-2024,16-05-2028,First Class,102778,United States,Monroe,North Carolina,28110,Chocolate,Gulf,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,3.75,1,2.45,1.3
5114,5115,US-2023-153577-CHO-TRI-54000,27-06-2025,21-12-2028,Standard Class,153577,United States,Highland Park,Illinois,60035,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.9
6505,6506,US-2023-161025-CHO-FUD-51000,03-12-2025,31-05-2029,Standard Class,161025,United States,Los Angeles,California,90049,Chocolate,Pacific,CHO-FUD-51000,Wonka Bar - Fudge Mallows,14.40,4,9.60,4.8
10072,10073,US-2024-155376-CHO-SCR-58000,22-12-2025,19-06-2030,Standard Class,155376,United States,Independence,Missouri,64055,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.3


## Initial Raw Data Observations

Based on the initial inspection of the raw dataset, the following observations are expected:

- The dataset contains order-level shipment records
- Key logistics columns such as Order Date, Ship Date, Ship Mode, Region, and State are present
- A direct Factory column is not available and will need to be created using the provided product-to-factory mapping
- Date fields are currently in raw string format and require validation
- Shipment date quality must be examined carefully because unrealistic lead times may exist
- Geographic fields may require standardization before route analysis
- This dataset appears suitable for route-level logistics analysis after proper cleaning and feature engineering

In [15]:
# Display a few important raw columns for quick manual check
important_columns = [
    "Order ID", "Order Date", "Ship Date", "Ship Mode",
    "City", "State/Province", "Region", "Division", "Product Name"
]

ncd[important_columns].head(10)

,Order ID,Order Date,Ship Date,Ship Mode,City,State/Province,Region,Division,Product Name
0,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,Houston,Texas,Interior,Chocolate,Wonka Bar - Milk Chocolate
1,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,Naperville,Illinois,Interior,Chocolate,Wonka Bar - Triple Dazzle Caramel
2,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,Naperville,Illinois,Interior,Chocolate,Wonka Bar - Nutty Crunch Surprise
3,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,Naperville,Illinois,Interior,Chocolate,Wonka Bar -Scrumdiddlyumptious
4,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,Philadelphia,Pennsylvania,Atlantic,Chocolate,Wonka Bar - Triple Dazzle Caramel
5,US-2021-167199-CHO-SCR-58000,06-01-2024,03-07-2026,Standard Class,Henderson,Kentucky,Gulf,Chocolate,Wonka Bar -Scrumdiddlyumptious
6,US-2021-167199-CHO-TRI-54000,06-01-2024,03-07-2026,Standard Class,Henderson,Kentucky,Gulf,Chocolate,Wonka Bar - Triple Dazzle Caramel
7,US-2021-106054-CHO-MIL-31000,06-01-2024,30-06-2026,First Class,Athens,Georgia,Gulf,Chocolate,Wonka Bar - Milk Chocolate
8,US-2021-167199-CHO-NUT-13000,06-01-2024,03-07-2026,Standard Class,Henderson,Kentucky,Gulf,Chocolate,Wonka Bar - Nutty Crunch Surprise
9,US-2021-167199-CHO-MIL-31000,06-01-2024,03-07-2026,Standard Class,Henderson,Kentucky,Gulf,Chocolate,Wonka Bar - Milk Chocolate


## Notebook 01 Conclusion

In this notebook, we completed the foundation stage of the project:

- understood the business problem
- defined the project objective
- loaded the raw dataset
- inspected dataset structure
- reviewed columns, data types, missing values, and duplicates
- documented initial raw data issues

### Next Step
The next notebook will focus on **Data Cleaning & Validation**, where we will:
- validate date formats
- inspect shipment date issues
- calculate lead time safely
- remove invalid or unrealistic lead-time records
- handle missing records
- standardize geographic fields